In [1]:
import os

In [2]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project'

In [9]:
import dagshub
dagshub.init(repo_owner='divyadarshan.dsa', repo_name='Kidney-Disease-Classification-Deep-Learning-Project', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as divyadarshan.dsa

Initialized MLflow to track repo "divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project"

Repository divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project initialized!

2026/09/11 02:44:45 INFO mlflow.tracking._tracking_service.client: 🏃 View run thoughtful-frog-421 at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/09c222874ca54d9781e199324306075a.
2026/09/11 02:44:45 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0.


In [7]:
%pip install dagshub

  Using cached dagshub-0.5.10-py3-none-any.whl.metadata (12 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached gitpython-3.1.62-py3-none-any.whl.metadata (13 kB)
  Using cached rich-14.3.4-py3-none-any.whl.metadata (18 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached tenacity-9.0.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached gql-4.0.0-py3-none-any.whl.metadata (10 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pandas-2.0.3-cp38-cp38-win_amd64.whl.metadata (18 kB)
  Using cached treelib-1.8.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pathvalidate-3.2.1-py3-none-any.whl.metadata (12 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached boto3-1.37.38-py3-none-any.whl.metadata (6.7 kB)
  Using cac

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.13.2 which is incompatible.


In [8]:
%pip install mlflow

  Using cached mlflow-2.17.2-py3-none-any.whl.metadata (29 kB)
  Using cached mlflow_skinny-2.17.2-py3-none-any.whl.metadata (30 kB)
  Using cached flask-3.0.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.14.1-py3-none-any.whl.metadata (7.4 kB)
  Using cached docker-7.2.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached matplotlib-3.7.5-cp38-cp38-win_amd64.whl.metadata (5.8 kB)
  Using cached pyarrow-17.0.0-cp38-cp38-win_amd64.whl.metadata (3.4 kB)
  Using cached scikit_learn-1.3.2-cp38-cp38-win_amd64.whl.metadata (11 kB)
  Using cached sqlalchemy-2.0.52-cp38-cp38-win_amd64.whl.metadata (9.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached waitress-3.0.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.102.0-py3-none-an

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.13.2 which is incompatible.


In [10]:
import tensorflow as tf

In [11]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [13]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [14]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/entbappy/Kidney-Disease-Classification-MLflow-DVC.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [15]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [16]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1. / 255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()

        # Changed model.evaluate() to self.model.evaluate()
        self.score = self.model.evaluate(self.valid_generator)

        self.save_score()

    def save_score(self):
        scores = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

    def log_into_mlflow(self):

        with mlflow.start_run():

            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics(
                {
                    "loss": self.score[0],
                    "accuracy": self.score[1]
                }
            )

            # Log the model without registering it
            mlflow.keras.log_model(
                self.model,
                "model"
            )

In [17]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-09-11 02:45:37,524: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-11 02:45:37,526: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-11 02:45:37,526: INFO: common: created directory at: artifacts]
Found 2207 images belonging to 2 classes.
138/138 [==============================] - 667s 5s/step - loss: 2.8687 - accuracy: 0.6964
[2026-09-11 02:56:46,282: INFO: common: json file saved at: scores.json]


2026/09/11 02:56:48 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\PC\AppData\Local\Temp\tmpentmd_27\model\data\model\assets
[2026-09-11 02:56:52,501: INFO: builder_impl: Assets written to: C:\Users\PC\AppData\Local\Temp\tmpentmd_27\model\data\model\assets]


2026/09/11 02:57:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/09/11 02:58:30 INFO mlflow.tracking._tracking_service.client: 🏃 View run popular-pug-503 at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/1f4fd1710d37433794caedfebed1840c.
2026/09/11 02:58:30 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/divyadarshan.dsa/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0.
